# Phase 0 - Fit the feature scalers  —  Rotax_915_iS

Run this FIRST, once per engine, before any training. Every other notebook loads
`scaler_<key>.pkl` through the data pipeline, and none of them can run without it.

The scaler is fit on the **train split only** - fitting on val/test would leak.
It covers exactly `tf_data_pipeline.FEATURE_COLS`, in that order; nothing else is
scaled (flags, stress, failure modes and the RUL label are all raw).

In [1]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, tensorflow as tf
from tensorflow import keras
import tf_data_pipeline as P, model_architectures as A, train_common as C

C.report_gpu()

  TF 2.16.2  devices ['CPU', 'GPU']


['CPU', 'GPU']

In [2]:
import glob, joblib, pandas as pd
from sklearn.preprocessing import StandardScaler

ENGINE = "Rotax_915_iS"
dd  = C.data_dir(ENGINE)
idx = json.load(open(C.index_path(ENGINE)))
train_files = sorted({e["file"] for e in idx["train"]})
print(f"{ENGINE}: {len(train_files)} train chunk(s)")

Rotax_915_iS: 6 train chunk(s)


In [3]:
# Fit incrementally so an 8M-row train split never has to be in memory at once.

sc = StandardScaler()

for f in train_files:

    df = pd.read_parquet(os.path.join(dd, f), columns=P.FEATURE_COLS)

    sc.partial_fit(df)

    print(f"  {f}: {len(df):,} rows")

    del df



out = C.scaler_path(ENGINE)

joblib.dump(sc, out)

print(f"\nwrote {out}")

print(f"  features {len(P.FEATURE_COLS)}   mean[0:3] {sc.mean_[:3].round(3)}")

  train_chunk_01.parquet: 1,506,780 rows
  train_chunk_02.parquet: 1,509,546 rows
  train_chunk_03.parquet: 1,501,636 rows
  train_chunk_04.parquet: 1,501,100 rows
  train_chunk_05.parquet: 1,507,038 rows
  train_chunk_06.parquet: 400,268 rows

wrote /Users/harsh/Documents/UAV_Engine/validation/models/scaler_915.pkl
  features 25   mean[0:3] [3.992683e+03 4.760000e-01 4.132900e+01]


In [4]:
# Sanity: the pipeline must now load end to end.
ds = P.make_dataset(C.index_path(ENGINE), dd, "val", out, batch_size=4)
x, y = next(iter(ds))
print("x        ", tuple(x["x"].shape), "expect (4,128,%d)" % P.N_FEATURES)
print("x_rul_aux", tuple(x["x_rul_aux"].shape), "expect (4,%d)" % P.N_RUL_AUX)
for k, v in y.items(): print(f"  {k:16s} {tuple(v.shape)}")
print("\nRUL sample (hours):", [round(float(v),1) for v in y["y_rul_hours"]])

2026-09-12 11:29:40.244735: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-12 11:29:40.245045: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-12 11:29:40.246060: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-12 11:29:40.246667: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-12 11:29:40.247051: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


x         (4, 128, 25) expect (4,128,25)
x_rul_aux (4, 10) expect (4,10)
  y_detection      (4,)
  y_diagnosis      (4, 8)
  y_severity       (4, 8)
  y_failure_mode   (4, 4)
  y_rul_hours      (4,)
  rul_valid        (4,)

RUL sample (hours): [833.0, 988.0, 75.2, 173.7]
